# Crawler BRACIS (Essencial)

Coleta trabalhos das edicoes de 2023, 2024 e 2025 do BRACIS no SOL/SBC e consolida em `DataFrame`.

In [1]:
import re
import time
from dataclasses import dataclass
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://sol.sbc.org.br/index.php"
BRACIS_SLUG = "bracis"
ANOS_ALVO = {2023, 2024, 2025}
REQUEST_TIMEOUT = 30
SLEEP_SECONDS = 0.2
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
)


In [2]:
@dataclass(frozen=True)
class Edicao:
    ano: int
    titulo: str
    link: str


def baixar_soup(url: str, session: requests.Session) -> BeautifulSoup:
    resposta = session.get(url, timeout=REQUEST_TIMEOUT)
    resposta.raise_for_status()
    return BeautifulSoup(resposta.text, "html.parser")


def extrair_ano(texto: str):
    if not texto:
        return None
    match = re.search(r"\b(20\d{2})\b", texto)
    return int(match.group(1)) if match else None


def listar_edicoes_bracis(session: requests.Session, anos_alvo):
    archive_url = f"{BASE_URL}/{BRACIS_SLUG}/issue/archive"
    soup = baixar_soup(archive_url, session)
    edicoes = []

    for bloco in soup.select("div.obj_issue_summary"):
        tag_titulo = bloco.select_one("a.title")
        if not tag_titulo:
            continue

        titulo_edicao = tag_titulo.get_text(" ", strip=True)
        ano_edicao = extrair_ano(titulo_edicao)

        if ano_edicao not in anos_alvo:
            continue

        link_edicao = urljoin(archive_url, tag_titulo.get("href", "").strip())
        edicoes.append(Edicao(ano=ano_edicao, titulo=titulo_edicao, link=link_edicao))

    edicoes.sort(key=lambda item: (item.ano, item.titulo), reverse=True)
    return edicoes


def listar_artigos_edicao(url_edicao: str, session: requests.Session):
    soup = baixar_soup(url_edicao, session)
    artigos = []
    links_vistos = set()

    for bloco in soup.select("div.obj_article_summary"):
        tag_artigo = bloco.select_one("div.title a")
        if not tag_artigo:
            continue

        link_artigo = urljoin(url_edicao, tag_artigo.get("href", "").strip())
        if not link_artigo or link_artigo in links_vistos:
            continue

        links_vistos.add(link_artigo)
        artigos.append(
            {
                "titulo_edicao": tag_artigo.get_text(" ", strip=True),
                "link_artigo": link_artigo,
            }
        )

    return artigos


def extrair_titulo_resumo(url_artigo: str, titulo_fallback: str, session: requests.Session):
    soup = baixar_soup(url_artigo, session)

    tag_titulo = soup.select_one("h1.page_title")
    titulo = tag_titulo.get_text(" ", strip=True) if tag_titulo else titulo_fallback

    resumo = ""
    tag_resumo = soup.select_one("div.item.abstract")
    if tag_resumo:
        tag_label = tag_resumo.select_one("h3.label")
        if tag_label:
            tag_label.decompose()
        resumo = tag_resumo.get_text(" ", strip=True)

    return titulo, resumo


def coletar_trabalhos_bracis(anos_alvo=ANOS_ALVO, sleep_seconds=SLEEP_SECONDS):
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    registros = []
    edicoes = listar_edicoes_bracis(session, anos_alvo)

    print(f"Edicoes BRACIS encontradas para {sorted(anos_alvo)}: {len(edicoes)}")

    for edicao in edicoes:
        print(f"- {edicao.ano}: {edicao.titulo}")
        artigos = listar_artigos_edicao(edicao.link, session)
        print(f"  Artigos na edicao: {len(artigos)}")

        total_artigos = len(artigos)
        for indice, artigo in enumerate(artigos, start=1):
            time.sleep(sleep_seconds)
            try:
                titulo, resumo = extrair_titulo_resumo(
                    artigo["link_artigo"], artigo["titulo_edicao"], session
                )
            except requests.RequestException as erro:
                print(f"  [aviso] erro em {artigo['link_artigo']}: {erro}")
                continue

            registros.append(
                {
                    "Conferencia": "BRACIS",
                    "Ano": edicao.ano,
                    "Edicao": edicao.titulo,
                    "Titulo": titulo,
                    "Resumo": resumo,
                    "Link": artigo["link_artigo"],
                }
            )

            if indice % 25 == 0 or indice == total_artigos:
                print(f"  Progresso: {indice}/{total_artigos}")

    df = pd.DataFrame(
        registros,
        columns=["Conferencia", "Ano", "Edicao", "Titulo", "Resumo", "Link"],
    )

    if not df.empty:
        df = (
            df.drop_duplicates(subset=["Ano", "Titulo", "Link"])
            .sort_values(["Ano", "Titulo"], ascending=[False, True])
            .reset_index(drop=True)
        )

    return df


In [3]:
df_bracis = coletar_trabalhos_bracis(anos_alvo={2023, 2024, 2025}, sleep_seconds=0.2)
df_bracis.head(10)

Edicoes BRACIS encontradas para [2023, 2024, 2025]: 3
- 2025: 2025: Anais da XXXV Brazilian Conference on Intelligent Systems
  Artigos na edicao: 148
  Progresso: 25/148
  Progresso: 50/148
  Progresso: 75/148
  Progresso: 100/148
  Progresso: 125/148
  Progresso: 148/148
- 2024: 2024: Anais da XXXIV Brazilian Conference on Intelligent Systems
  Artigos na edicao: 116
  Progresso: 25/116
  Progresso: 50/116
  Progresso: 75/116
  Progresso: 100/116
  Progresso: 116/116
- 2023: 2023: Anais da XII Brazilian Conference on Intelligent Systems
  Artigos na edicao: 90
  Progresso: 25/90
  Progresso: 50/90
  [aviso] erro em https://sol.sbc.org.br/index.php/bracis/article/view/28414: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  Progresso: 75/90
  Progresso: 90/90


,Conferencia,Ano,Edicao,Titulo,Resumo,Link
0,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Categorical Kalman Filter for Human Activity...,Human Activity Recognition (HAR) has gained in...,https://sol.sbc.org.br/index.php/bracis/articl...
1,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Comparative Study of Machine Learning Models...,The bee flora refers to the group of plants fr...,https://sol.sbc.org.br/index.php/bracis/articl...
2,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Comprehensive Study of Fitness Landscapes in...,The search space in Automated Machine Learning...,https://sol.sbc.org.br/index.php/bracis/articl...
3,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Double Transfer Learning Approach for Improv...,Cancer is one of the most devastating health c...,https://sol.sbc.org.br/index.php/bracis/articl...
4,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Fully Automatic Approach for COVID-19 Diagno...,The pandemic caused by the COVID-19 virus has ...,https://sol.sbc.org.br/index.php/bracis/articl...
5,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Generative Domain Adaptation Scheme for Swif...,Deep learning models have demonstrated remarka...,https://sol.sbc.org.br/index.php/bracis/articl...
6,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Livestock Monitoring System and Machine Lear...,Timely disease diagnosis is essential for redu...,https://sol.sbc.org.br/index.php/bracis/articl...
7,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Multi-Dimensional Comparative Study of Gener...,Synthetic data is increasingly important in pr...,https://sol.sbc.org.br/index.php/bracis/articl...
8,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Multidimensional Analysis of Swarm Dynamics ...,Traditional assessments of metaheuristics typi...,https://sol.sbc.org.br/index.php/bracis/articl...
9,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A NAS-Optimized Deep Learning Model for Custom...,Deep Learning (DL) methods offer competitive a...,https://sol.sbc.org.br/index.php/bracis/articl...


In [4]:
print(f"Total de registros: {len(df_bracis)}")
print("\nColunas:", list(df_bracis.columns))
print("\nTrabalhos por ano:")
print(df_bracis["Ano"].value_counts().sort_index(ascending=False))

Total de registros: 353

Colunas: ['Conferencia', 'Ano', 'Edicao', 'Titulo', 'Resumo', 'Link']

Trabalhos por ano:
Ano
2025    148
2024    116
2023     89
Name: count, dtype: int64


In [5]:
arquivo_saida = "bracis_trabalhos_2023_2025.csv"
df_bracis.to_csv(arquivo_saida, index=False)
print(f"CSV salvo em: {arquivo_saida}")

CSV salvo em: bracis_trabalhos_2023_2025.csv


## Parte 1 - Bag of Words, TF-IDF e Modelagem de Topicos

Secao para a atividade:

- (a) representacoes BoW e TF-IDF;
- (b) modelagem de topicos com NMF (e LDA opcional);
- (c) interpretacao dos topicos.

In [8]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

pd.set_option("display.max_colwidth", 220)

In [9]:
if "df_bracis" not in globals() or df_bracis is None or len(df_bracis) == 0:
    df_bracis = pd.read_csv("bracis_trabalhos_2023_2025.csv")

df_nlp = df_bracis.copy()
for coluna in ["Titulo", "Resumo"]:
    df_nlp[coluna] = (
        df_nlp[coluna]
        .fillna("")
        .astype(str)
        .str.replace(r"\\s+", " ", regex=True)
        .str.strip()
    )

df_nlp["texto"] = (df_nlp["Titulo"] + ". " + df_nlp["Resumo"]).str.strip()
df_nlp = df_nlp[df_nlp["texto"].str.len() > 0].reset_index(drop=True)

print(f"Documentos no corpus: {len(df_nlp)}")
df_nlp[["Ano", "Titulo", "Resumo"]].head(3)

Documentos no corpus: 353


,Ano,Titulo,Resumo
0,2025,A Categorical Kalman Filter for Human Activity Recognition,"Human Activity Recognition (HAR) has gained increasing attention in recent years due to its wide range of applications in health, sports, and surveillance. The ubiquity and unobtrusive nature of inertial measurement ..."
1,2025,A Comparative Study of Machine Learning Models for Bee Flora Classification: Integrating Feature Extractors with Classifiers,"The bee flora refers to the group of plants from which bees collect floral resources, such as pollen, nectar, or both. The potential for beekeeping production in a region is influenced by its floral coverage. Therefo..."
2,2025,A Comprehensive Study of Fitness Landscapes in AutoML: The Impact of Dataset Complexity and Search Space Neutrality,The search space in Automated Machine Learning (AutoML) consists of numerous combinations of machine learning pipelines. Understanding how optimal solutions are distributed within this space is key to improving searc...


In [10]:
# (a) Bag of Words
bow_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
)

X_bow = bow_vectorizer.fit_transform(df_nlp["texto"])
vocab_bow = bow_vectorizer.get_feature_names_out()

freq_bow = np.asarray(X_bow.sum(axis=0)).ravel()
idx_top_bow = freq_bow.argsort()[::-1][:25]
bow_top_terms = pd.DataFrame(
    {
        "termo": vocab_bow[idx_top_bow],
        "frequencia": freq_bow[idx_top_bow],
    }
)

print("Matriz BoW:", X_bow.shape)
print("Vocabulario BoW:", len(vocab_bow))
bow_top_terms

Matriz BoW: (353, 3123)
Vocabulario BoW: 3123


,termo,frequencia
0,models,429
1,data,422
2,learning,375
3,based,364
4,model,360
5,results,291
6,using,280
7,performance,248
8,approach,233
9,classification,200


In [11]:
# (a) TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

X_tfidf = tfidf_vectorizer.fit_transform(df_nlp["texto"])
vocab_tfidf = tfidf_vectorizer.get_feature_names_out()

mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
idx_top_tfidf = mean_tfidf.argsort()[::-1][:25]
tfidf_top_terms = pd.DataFrame(
    {
        "termo": vocab_tfidf[idx_top_tfidf],
        "peso_medio_tfidf": mean_tfidf[idx_top_tfidf],
    }
)

print("Matriz TF-IDF:", X_tfidf.shape)
print("Vocabulario TF-IDF:", len(vocab_tfidf))
tfidf_top_terms

Matriz TF-IDF: (353, 3123)
Vocabulario TF-IDF: 3123


,termo,peso_medio_tfidf
0,models,0.030369
1,data,0.029334
2,learning,0.028983
3,based,0.028145
4,model,0.027823
5,using,0.024292
6,results,0.023846
7,performance,0.023059
8,approach,0.022839
9,classification,0.021393


In [12]:
def top_words_by_topic(components, feature_names, n_top_words=12):
    linhas = []
    for topic_id, topic_weights in enumerate(components):
        top_idx = topic_weights.argsort()[::-1][:n_top_words]
        palavras = [feature_names[i] for i in top_idx]
        linhas.append(
            {
                "topico": topic_id,
                "palavras_chave": ", ".join(palavras),
            }
        )
    return pd.DataFrame(linhas)


In [13]:
# (b) NMF sobre TF-IDF (preferencial)
N_TOPICS = 8
N_TOP_WORDS = 12

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=42,
    init="nndsvda",
    max_iter=600,
)

W_nmf = nmf_model.fit_transform(X_tfidf)
H_nmf = nmf_model.components_

nmf_topics_df = top_words_by_topic(H_nmf, vocab_tfidf, n_top_words=N_TOP_WORDS)
df_nlp["topico_nmf"] = W_nmf.argmax(axis=1)
df_nlp["score_topico_nmf"] = W_nmf.max(axis=1)

print("Erro de reconstrucao (NMF):", round(nmf_model.reconstruction_err_, 4))
nmf_topics_df

Erro de reconstrucao (NMF): 17.7512


,topico,palavras_chave
0,0,"machine, machine learning, learning, study, models, explainable, support, explanations, analysis, artificial, based, social"
1,1,"language, portuguese, language models, legal, models, large, llms, fine, brazilian, large language, brazilian portuguese, domain"
2,2,"images, diagnosis, segmentation, cancer, deep, detection, image, deep learning, classification, accuracy, method, convolutional neural"
3,3,"optimization, objective, algorithm, algorithms, multi objective, multi, genetic, evolutionary, search, problem, solutions, space"
4,4,"data, training, supervised, label, classification, instances, labeled, learning, unlabeled, datasets, self, dataset"
5,5,"series, time series, time, forecasting, lstm, short term, term, neural, short, term memory, long short, memory"
6,6,"graph, graph neural, networks, nodes, network, information, graphs, node, neural, neural networks, epidemic, data"
7,7,"agent, reinforcement learning, reinforcement, states, goal, probability, policy, dead, criterion, criteria, ssps, state"


In [14]:
# (b) LDA opcional sobre BoW
lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method="batch",
    max_iter=25,
)

W_lda = lda_model.fit_transform(X_bow)
H_lda = lda_model.components_

lda_topics_df = top_words_by_topic(H_lda, vocab_bow, n_top_words=N_TOP_WORDS)
df_nlp["topico_lda"] = W_lda.argmax(axis=1)

print("Perplexidade (LDA):", round(lda_model.perplexity(X_bow), 2))
lda_topics_df

Perplexidade (LDA): 1909.18


,topico,palavras_chave
0,0,"language, text, models, large, legal, llms, large language, language models, generation, corpus, retrieval, llm"
1,1,"data, model, models, brazilian, based, classification, methods, results, different, dataset, network, analysis"
2,2,"data, based, learning, models, model, graph, using, methods, time, performance, results, series"
3,3,"learning, models, data, model, machine, machine learning, based, time, using, approach, performance, results"
4,4,"models, model, language, performance, fine, portuguese, results, data, language models, tuning, fine tuning, based"
5,5,"data, learning, based, method, models, instances, training, new, approach, results, model, propose"
6,6,"images, learning, classification, using, accuracy, detection, results, approach, deep, model, segmentation, neural"
7,7,"multi, optimization, algorithms, objective, algorithm, problem, problems, based, label, states, search, multi label"


In [15]:
# (c) Interpretacao dos topicos (NMF)
def interpretar_topicos_nmf(df_docs, topicos_df, top_n_titulos=3):
    linhas = []
    for _, linha in topicos_df.iterrows():
        t = int(linha["topico"])
        sub = df_docs[df_docs["topico_nmf"] == t].copy()

        qtd_docs = len(sub)
        por_ano = sub["Ano"].value_counts().sort_index().to_dict()
        titulos_rep = (
            sub.sort_values("score_topico_nmf", ascending=False)["Titulo"]
            .head(top_n_titulos)
            .tolist()
        )

        termos = linha["palavras_chave"].split(", ")[:4]
        interpretacao = (
            f"Topico {t} foca em {', '.join(termos)}. "
            f"Documentos: {qtd_docs}. Distribuicao por ano: {por_ano}."
        )

        linhas.append(
            {
                "topico": t,
                "palavras_chave": linha["palavras_chave"],
                "qtd_documentos": qtd_docs,
                "distribuicao_anos": por_ano,
                "titulos_representativos": " | ".join(titulos_rep),
                "interpretacao_inicial": interpretacao,
            }
        )

    return pd.DataFrame(linhas).sort_values("qtd_documentos", ascending=False)


interpretacao_nmf_df = interpretar_topicos_nmf(df_nlp, nmf_topics_df)
interpretacao_nmf_df

,topico,palavras_chave,qtd_documentos,distribuicao_anos,titulos_representativos,interpretacao_inicial
1,1,"language, portuguese, language models, legal, models, large, llms, fine, brazilian, large language, brazilian portuguese, domain",67,"{2023: 11, 2024: 29, 2025: 27}",LegalBert-pt: A Pretrained Language Model for the Brazilian Portuguese Legal Domain | An Ensemble of LLMs Finetuned with LoRA for NER in Portuguese Legal Documents | Generative SLMs Meet Brazilian Legal Documents: Ef...,"Topico 1 foca em language, portuguese, language models, legal. Documentos: 67. Distribuicao por ano: {2023: 11, 2024: 29, 2025: 27}."
2,2,"images, diagnosis, segmentation, cancer, deep, detection, image, deep learning, classification, accuracy, method, convolutional neural",64,"{2023: 18, 2024: 17, 2025: 29}",DWNNet-Therm: A Deep Wavelet Neural Network Architecture Dedicated to Multiclass Classification in Breast Thermography | Deep Learning for Helicobacter Pylori Classification: A Comparative Approach Using Transfer Lea...,"Topico 2 foca em images, diagnosis, segmentation, cancer. Documentos: 64. Distribuicao por ano: {2023: 18, 2024: 17, 2025: 29}."
4,4,"data, training, supervised, label, classification, instances, labeled, learning, unlabeled, datasets, self, dataset",52,"{2023: 11, 2024: 13, 2025: 28}",Semi-supervised Predictive Clustering Trees for Multi-label Protein Subcellular Localization | Pseudo-labeling for Multi-label Legal Text Classification | An Instance Level Analysis of Classification Difficulty for U...,"Topico 4 foca em data, training, supervised, label. Documentos: 52. Distribuicao por ano: {2023: 11, 2024: 13, 2025: 28}."
0,0,"machine, machine learning, learning, study, models, explainable, support, explanations, analysis, artificial, based, social",49,"{2023: 10, 2024: 16, 2025: 23}",Alzheimer’s Disease Neuroimaging Initiative Comparing LIME and SHAP Global Explanations for Human Activity Recognition | Integrating Explainable AI with Species Distribution Models to Assess Anthropogenic Impacts on ...,"Topico 0 foca em machine, machine learning, learning, study. Documentos: 49. Distribuicao por ano: {2023: 10, 2024: 16, 2025: 23}."
6,6,"graph, graph neural, networks, nodes, network, information, graphs, node, neural, neural networks, epidemic, data",34,"{2023: 8, 2024: 12, 2025: 14}",Deep Graph Clustering Using Graph Neural Networks and Seed Detection | Enhancing Graph Data Quality by Leveraging Heterogeneous Node Features and Embeddings | Detecting Multiple Epidemic Sources in Network Epidemics ...,"Topico 6 foca em graph, graph neural, networks, nodes. Documentos: 34. Distribuicao por ano: {2023: 8, 2024: 12, 2025: 14}."
7,7,"agent, reinforcement learning, reinforcement, states, goal, probability, policy, dead, criterion, criteria, ssps, state",31,"{2023: 13, 2024: 13, 2025: 5}",Reinforcement Learning with Utility-Based Semantic for Goals | Dead-End Discovery and Secure Exploration via Large Language Models | α-MCMP: Trade-Offs Between Probability and Cost in SSPs with the MCMP Criterion,"Topico 7 foca em agent, reinforcement learning, reinforcement, states. Documentos: 31. Distribuicao por ano: {2023: 13, 2024: 13, 2025: 5}."
3,3,"optimization, objective, algorithm, algorithms, multi objective, multi, genetic, evolutionary, search, problem, solutions, space",29,"{2023: 12, 2024: 8, 2025: 9}","Special-Crowd-Distance Boosted MESH Applied to the Operation of Cascade Hydro-Power Plants | Impact of Parent Selection Operator on the FDEA Algorithm | Maximum Dispersion, Maximum Concentration: Enhancing the Qualit...","Topico 3 foca em optimization, objective, algorithm, algorithms. Documentos: 29. Distribuicao por ano: {2023: 12, 2024: 8, 2025: 9}."
5,5,"series, time series, time, forecasting, lstm, short term, term, neural, short, term memory, long short, memory",27,"{2023: 6, 2024: 8, 2025: 13}",AutoFAR: An Intelligent Hybrid System for Boosting Autoformer Forecasts | Increasing Time Series Forecasting Performance with Dyna

### Texto para relatorio (sugestao)

Use a tabela `interpretacao_nmf_df` para descrever cada topico no relatorio.

Estrutura sugerida:

1. Palavras-chave do topico.
2. Quantidade de trabalhos e distribuicao por ano.
3. Titulos representativos.
4. Interpretacao semantica final (refinada por voce ou por LLM).

## Parte 2 - Metodologia do artigo-base (Embeddings + UMAP + HDBSCAN + c-TF-IDF)

Implementacao alinhada ao artigo-base (`s41598-025-92190-7`):

- Embeddings contextuais com `all-MiniLM-L6-v2`;
- reducao de dimensionalidade com UMAP (`n_neighbors=15`, `random_state=100`);
- agrupamento com HDBSCAN e tratamento de ruido/outliers;
- representacao de topicos via c-TF-IDF com `n-gram` de `(1, 2)` e `top_n_words=20`;
- atribuicao probabilistica de topicos (`calculate_probabilities=True`);
- validacao com coerencia (`c_v`), distancia intertopicos e inspecao manual.


Dependencias da Parte 2 (rode se necessario):

```python
%pip install -q bertopic sentence-transformers umap-learn hdbscan gensim
```


In [ ]:
import warnings
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    from umap import UMAP
    from hdbscan import HDBSCAN
except ImportError as exc:
    raise ImportError(
        "Dependencias ausentes para a Parte 2. Rode: %pip install -q bertopic sentence-transformers umap-learn hdbscan gensim"
    ) from exc

warnings.filterwarnings("ignore")


In [ ]:
if "df_nlp" not in globals() or df_nlp is None or len(df_nlp) == 0:
    if "df_bracis" not in globals() or df_bracis is None or len(df_bracis) == 0:
        df_bracis = pd.read_csv("bracis_trabalhos_2023_2025.csv")

    df_nlp = df_bracis.copy()
    for coluna in ["Titulo", "Resumo"]:
        df_nlp[coluna] = (
            df_nlp[coluna]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    df_nlp["texto"] = (df_nlp["Titulo"] + ". " + df_nlp["Resumo"]).str.strip()
    df_nlp = df_nlp[df_nlp["texto"].str.len() > 0].reset_index(drop=True)

docs_part2 = df_nlp["texto"].tolist()

stopwords_part2 = set(ENGLISH_STOP_WORDS).union({"use", "add", "related"})

def tokenize_simple(texto):
    tokens = re.findall(r"[a-zA-Z][a-zA-Z\-]{1,}", str(texto).lower())
    return [t for t in tokens if t not in stopwords_part2 and len(t) >= 3]

tokenized_docs_part2 = [tokenize_simple(doc) for doc in docs_part2]
dictionary_part2 = Dictionary(tokenized_docs_part2)
corpus_part2 = [dictionary_part2.doc2bow(tokens) for tokens in tokenized_docs_part2]

print(f"Documentos para Parte 2: {len(docs_part2)}")


In [ ]:
def extrair_topic_terms_bertopic(topic_model, n_top_words=20):
    termos = []
    for topic_id in sorted(topic_model.get_topics().keys()):
        if topic_id == -1:
            continue
        palavras = topic_model.get_topic(topic_id) or []
        termos_topic = [w for w, _ in palavras[:n_top_words]]
        if termos_topic:
            termos.append(termos_topic)
    return termos

def coherence_cv(topic_terms, tokenized_docs, dictionary):
    topic_terms = [t for t in topic_terms if t]
    if not topic_terms:
        return float("nan")

    cm = CoherenceModel(
        topics=topic_terms,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v",
    )
    return float(cm.get_coherence())

def intertopic_distance_min(topic_model):
    info = topic_model.get_topic_info()
    topicos_validos = info[info["Topic"] != -1]["Topic"].tolist()

    if len(topicos_validos) < 2:
        return float("nan")

    embeddings = topic_model.topic_embeddings_[topicos_validos]
    sim = cosine_similarity(embeddings)
    dist = 1 - sim
    np.fill_diagonal(dist, np.nan)
    return float(np.nanmin(dist))

def topic_diversity(topic_terms, topk=10):
    cortes = [termos[:topk] for termos in topic_terms if termos]
    if not cortes:
        return float("nan")
    total = sum(len(x) for x in cortes)
    unicos = len({w for linha in cortes for w in linha})
    return float(unicos / total) if total else float("nan")


In [ ]:
embedding_model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

embeddings_part2 = embedding_model.encode(
    docs_part2,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Embeddings gerados:", embeddings_part2.shape)


In [ ]:
avaliacoes_part2 = []
modelos_part2 = {}

for k in range(4, 17):
    umap_model = UMAP(n_neighbors=15, metric="cosine", random_state=100)
    hdbscan_model = HDBSCAN(
        min_cluster_size=20,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    vectorizer_model = CountVectorizer(
        stop_words=list(stopwords_part2),
        ngram_range=(1, 2),
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=20,
        min_topic_size=20,
        nr_topics=k,
        calculate_probabilities=True,
        verbose=False,
    )

    topics, probs = topic_model.fit_transform(docs_part2, embeddings_part2)

    info = topic_model.get_topic_info()
    n_topicos_validos = int((info["Topic"] != -1).sum())
    taxa_outlier = float((np.array(topics) == -1).mean())

    termos = extrair_topic_terms_bertopic(topic_model, n_top_words=20)
    coerencia_cv = coherence_cv(termos, tokenized_docs_part2, dictionary_part2)
    dist_min = intertopic_distance_min(topic_model)
    diversidade_10 = topic_diversity(termos, topk=10)

    avaliacoes_part2.append(
        {
            "k_alvo": k,
            "topicos_encontrados": n_topicos_validos,
            "coerencia_c_v": coerencia_cv,
            "distancia_intertopico_min": dist_min,
            "diversidade_top10": diversidade_10,
            "taxa_outlier": taxa_outlier,
        }
    )

    modelos_part2[k] = (topic_model, topics, probs)

avaliacoes_part2_df = pd.DataFrame(avaliacoes_part2).sort_values(
    ["coerencia_c_v", "distancia_intertopico_min", "diversidade_top10"],
    ascending=[False, False, False],
).reset_index(drop=True)

avaliacoes_part2_df


In [ ]:
# O artigo-base conclui em 5 topicos; mantemos essa decisao para reproducao metodologica.
k_escolhido_part2 = 5
topic_model_part2, topics_part2, probs_part2 = modelos_part2[k_escolhido_part2]

doc_info_part2 = topic_model_part2.get_document_info(docs_part2)

df_nlp["topico_bertopic"] = doc_info_part2["Topic"].values
df_nlp["score_topico_bertopic"] = doc_info_part2["Probability"].fillna(0).values

topic_info_part2 = topic_model_part2.get_topic_info().copy()
topic_info_part2


In [ ]:
def tabela_topicos_bertopic(topic_model, top_n_words=12):
    linhas = []
    info = topic_model.get_topic_info()

    for _, row in info.iterrows():
        topic_id = int(row["Topic"])
        if topic_id == -1:
            continue

        palavras = topic_model.get_topic(topic_id) or []
        palavras_top = [w for w, _ in palavras[:top_n_words]]

        linhas.append(
            {
                "topico": topic_id,
                "qtd_documentos": int(row["Count"]),
                "palavras_chave": ", ".join(palavras_top),
            }
        )

    return pd.DataFrame(linhas).sort_values("qtd_documentos", ascending=False)

topicos_part2_df = tabela_topicos_bertopic(topic_model_part2, top_n_words=12)
topicos_part2_df


In [ ]:
def interpretar_topicos_bertopic(df_docs, topicos_df, top_n_titulos=3):
    linhas = []

    for _, linha in topicos_df.iterrows():
        t = int(linha["topico"])
        sub = df_docs[df_docs["topico_bertopic"] == t].copy()

        qtd_docs = len(sub)
        por_ano = sub["Ano"].value_counts().sort_index().to_dict()
        titulos_rep = (
            sub.sort_values("score_topico_bertopic", ascending=False)["Titulo"]
            .head(top_n_titulos)
            .tolist()
        )

        termos = linha["palavras_chave"].split(", ")[:4]
        interpretacao = (
            f"Topico {t} foca em {', '.join(termos)}. "
            f"Documentos: {qtd_docs}. Distribuicao por ano: {por_ano}."
        )

        linhas.append(
            {
                "topico": t,
                "palavras_chave": linha["palavras_chave"],
                "qtd_documentos": qtd_docs,
                "distribuicao_anos": por_ano,
                "titulos_representativos": " | ".join(titulos_rep),
                "interpretacao_inicial": interpretacao,
            }
        )

    return pd.DataFrame(linhas).sort_values("qtd_documentos", ascending=False)

interpretacao_part2_df = interpretar_topicos_bertopic(df_nlp, topicos_part2_df)
interpretacao_part2_df


### Visualizacao UMAP 2D dos topicos (Parte 2)

Projecao dos embeddings em 2 dimensoes com UMAP para visualizar a separacao dos clusters do BERTopic/HDBSCAN.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if "embeddings_part2" not in globals():
    raise ValueError("Execute a celula de embeddings da Parte 2 antes deste grafico.")

if "topico_bertopic" not in df_nlp.columns:
    raise ValueError("Execute a celula de atribuicao de topicos BERTopic antes deste grafico.")

umap_vis_model = UMAP(
    n_neighbors=15,
    n_components=2,
    metric="cosine",
    random_state=100,
)

emb_2d_part2 = umap_vis_model.fit_transform(embeddings_part2)

plot_df = pd.DataFrame({
    "UMAP-1": emb_2d_part2[:, 0],
    "UMAP-2": emb_2d_part2[:, 1],
    "topico": df_nlp["topico_bertopic"].values,
})

plot_df["topico_label"] = plot_df["topico"].apply(
    lambda t: "Outlier (-1)" if int(t) == -1 else f"Topico {int(t)}"
)

ordem = sorted(plot_df["topico"].unique(), key=lambda x: (x == -1, x))
ordem_labels = ["Outlier (-1)" if int(t) == -1 else f"Topico {int(t)}" for t in ordem]

plt.figure(figsize=(11, 8))
sns.scatterplot(
    data=plot_df,
    x="UMAP-1",
    y="UMAP-2",
    hue="topico_label",
    hue_order=ordem_labels,
    palette="tab20",
    s=30,
    alpha=0.8,
    linewidth=0,
)
plt.title("UMAP 2D dos documentos BRACIS (colorido por topico BERTopic)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(title="Topicos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Parte 5 - Comparacao entre Parte 1 e Parte 2

Comparacao entre:

- Parte 1: BoW/TF-IDF + NMF/LDA
- Parte 2: BERTopic (Embeddings + UMAP + HDBSCAN + c-TF-IDF)


In [ ]:
def topic_terms_from_df(topics_df_col):
    return [str(v).split(", ") for v in topics_df_col.tolist() if str(v).strip()]

# Parte 1
nmf_terms = topic_terms_from_df(nmf_topics_df["palavras_chave"])
lda_terms = topic_terms_from_df(lda_topics_df["palavras_chave"])

coerencia_nmf = coherence_cv(nmf_terms, tokenized_docs_part2, dictionary_part2)
coerencia_lda = coherence_cv(lda_terms, tokenized_docs_part2, dictionary_part2)
diversidade_nmf = topic_diversity(nmf_terms, topk=10)
diversidade_lda = topic_diversity(lda_terms, topk=10)

# Parte 2
bertopic_terms = extrair_topic_terms_bertopic(topic_model_part2, n_top_words=20)
coerencia_bertopic = coherence_cv(bertopic_terms, tokenized_docs_part2, dictionary_part2)
diversidade_bertopic = topic_diversity(bertopic_terms, topk=10)
taxa_outlier_part2 = float((df_nlp["topico_bertopic"] == -1).mean())

comparacao_df = pd.DataFrame(
    [
        {
            "metodo": "NMF (Parte 1)",
            "n_topicos": int(nmf_topics_df["topico"].nunique()),
            "coerencia_c_v": coerencia_nmf,
            "diversidade_top10": diversidade_nmf,
            "cobertura_docs": 1.0,
            "metrica_extra": f"erro_reconstrucao={nmf_model.reconstruction_err_:.4f}",
        },
        {
            "metodo": "LDA (Parte 1)",
            "n_topicos": int(lda_topics_df["topico"].nunique()),
            "coerencia_c_v": coerencia_lda,
            "diversidade_top10": diversidade_lda,
            "cobertura_docs": 1.0,
            "metrica_extra": f"perplexidade={lda_model.perplexity(X_bow):.2f}",
        },
        {
            "metodo": "BERTopic (Parte 2)",
            "n_topicos": int((topic_info_part2["Topic"] != -1).sum()),
            "coerencia_c_v": coerencia_bertopic,
            "diversidade_top10": diversidade_bertopic,
            "cobertura_docs": 1.0 - taxa_outlier_part2,
            "metrica_extra": f"outlier_rate={taxa_outlier_part2:.3f}",
        },
    ]
).sort_values("coerencia_c_v", ascending=False).reset_index(drop=True)

comparacao_df


In [ ]:
melhor = comparacao_df.iloc[0]

print("Resumo comparativo:")
print(f"- Melhor coerencia c_v: {melhor['metodo']} ({melhor['coerencia_c_v']:.4f})")
print(
    f"- BERTopic: topicos={int((topic_info_part2['Topic'] != -1).sum())}, cobertura={(1.0 - taxa_outlier_part2):.2%}, outliers={taxa_outlier_part2:.2%}"
)
print(
    "- Interpretacao: Parte 1 tende a gerar topicos mais lexicais; Parte 2 tende a separar melhor semantica por usar embeddings contextuais."
)
